In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import functions

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

import re

In [2]:
df = pd.read_csv("../../data/clean_train.csv", sep=";",encoding="utf-8-sig", keep_default_na=False)

## Create Transcription Headers

In [3]:
df["transcription_headers"] = df["transcription"].apply(functions.extract_headers)
df["transcription_headers_model"] = df["transcription_headers"].apply(functions.headers_for_model)

# TF-IDF

### Train/Test Split

In [4]:
from sklearn.model_selection import train_test_split

train_indices, valid_indices = train_test_split(
    df.index,
    test_size=0.2,
    random_state=42,
    stratify=df["medical_specialty"],
)

y_train = df.loc[train_indices, "medical_specialty"]
y_valid = df.loc[valid_indices, "medical_specialty"]

### Classifiers

In [5]:
linear_svc = LinearSVC(C=0.5,class_weight="balanced",)
logistic_regression = LogisticRegression(C=1.0,class_weight="balanced",max_iter=2000,random_state=42,)
multinomial_nb = MultinomialNB()

### Testing Baseline

In [17]:
input_just_description = functions.build_input_variants(df, ["description"])
pd.DataFrame([functions.evaluate_input_extra(input_just_description, "Just description", multinomial_nb, "Multinomial NB", train_indices, valid_indices, y_train, y_valid)[1]])

,model,input,accuracy,macro_f1,lowest_f1,lowest_f1_label
0,Multinomial NB,Just description,0.384615,0.132751,0.0,Dermatology


### Testing Adding Headers To All

In [7]:
input_headers = functions.build_input_variants(df, ["description", "sample_name", "transcription", "keywords", "transcription_headers_model"])
input = functions.build_input_variants(df, ["description", "sample_name", "transcription", "keywords"])



results = pd.DataFrame([
    #functions.evaluate_input_extra(input_headers, "With Headers", linear_svc, "Linear SVC", train_indices, valid_indices, y_train, y_valid)[1],
    #functions.evaluate_input_extra(input, "Without Headers", linear_svc, "Linear SVC", train_indices, valid_indices, y_train, y_valid)[1],
    #functions.evaluate_input_extra(input_headers, "With Headers", logistic_regression, "Logistic Regression", train_indices, valid_indices, y_train, y_valid)[1],
    #functions.evaluate_input_extra(input, "Without Headers", logistic_regression, "Logistic Regression", train_indices, valid_indices, y_train, y_valid)[1],
    #functions.evaluate_input_extra(input_headers, "With Headers", multinomial_nb, "Multinomial NB", train_indices, valid_indices, y_train, y_valid)[1],
    #functions.evaluate_input_extra(input, "Without Headers", multinomial_nb, "Multinomial NB", train_indices, valid_indices, y_train, y_valid)[1]
])

results

""


### Keywords and Transcription Headers

Linear SVC 

Logistic Regression

Multinomial Naive Bayes

In [8]:
input = functions.build_input_variants(df, ["keywords", "transcription_headers_model"])
input_key = functions.build_input_variants(df, ["keywords"])

pred1, result1 = functions.evaluate_input_extra(input, "key and Headers", linear_svc, "Linear SVC", train_indices, valid_indices, y_train, y_valid)
pred2, result2 = functions.evaluate_input_extra(input_key, "key", linear_svc, "Linear SVC", train_indices, valid_indices, y_train, y_valid)
pred3, result3 = functions.evaluate_input_extra(input, "key and Headers", logistic_regression, "Log Regression", train_indices, valid_indices, y_train, y_valid)
pred4, result4 = functions.evaluate_input_extra(input_key, "key", logistic_regression, "Log Regression", train_indices, valid_indices, y_train, y_valid)
pred5, result5 = functions.evaluate_input_extra(input, "key and Headers", multinomial_nb, "multinomial_nb", train_indices, valid_indices, y_train, y_valid)
pred6, result6 = functions.evaluate_input_extra(input_key, "key", multinomial_nb, "multinomial_nb", train_indices, valid_indices, y_train, y_valid)

results = pd.DataFrame([result1, result2, result3, result4, result5, result6])
results

,model,input,accuracy,macro_f1,lowest_f1,lowest_f1_label
0,Linear SVC,key and Headers,0.879925,0.844047,0.705882,Psychiatry-Psychology
1,Linear SVC,key,0.840525,0.822564,0.166667,Psychiatry-Psychology
2,Log Regression,key and Headers,0.793621,0.752410,0.545455,Dermatology
3,Log Regression,key,0.782364,0.764078,0.166667,Psychiatry-Psychology
4,multinomial_nb,key and Headers,0.525328,0.303518,0.000000,Dermatology
5,multinomial_nb,key,0.512195,0.295635,0.000000,Dermatology


In [9]:
labels = sorted(y_valid.unique())

#functions.show_classification_report(y_valid,pred2,labels,)
#functions.show_classification_report(y_valid,pred4,labels,)
#functions.show_classification_report(y_valid,pred6,labels,)

In [16]:
input = functions.build_input_variants(df, ["keywords", "transcription_headers_model"])
input_desc = functions.build_input_variants(df, ["description", "keywords", "transcription_headers_model"])
input_desc_sample = functions.build_input_variants(df, ["description", "sample_name", "keywords", "transcription_headers_model"])
input_sample = functions.build_input_variants(df, ["sample_name", "keywords"])

results = pd.DataFrame([
    functions.evaluate_input_extra(input, "key and Headers", linear_svc, "Linear SVC", train_indices, valid_indices, y_train, y_valid)[1],
    functions.evaluate_input_extra(input_desc, "desc, key and Headers", linear_svc, "Linear SVC", train_indices, valid_indices, y_train, y_valid)[1],
    functions.evaluate_input_extra(input_desc_sample, "desc, sample, key and Headers", linear_svc, "Linear SVC", train_indices, valid_indices, y_train, y_valid)[1],
    functions.evaluate_input_extra(input_sample, "sample, key and Headers", linear_svc, "Linear SVC", train_indices, valid_indices, y_train, y_valid)[1],
    functions.evaluate_input_extra(input, "key and Headers", logistic_regression, "logistic regression", train_indices, valid_indices, y_train, y_valid)[1],
    functions.evaluate_input_extra(input_desc, "desc, key and Headers", logistic_regression, "logistic regression", train_indices, valid_indices, y_train, y_valid)[1],
    functions.evaluate_input_extra(input_desc_sample, "desc, sample, key and Headers", logistic_regression, "logistic regression", train_indices, valid_indices, y_train, y_valid)[1],
    functions.evaluate_input_extra(input_sample, "sample, key and Headers", logistic_regression, "logistic regression", train_indices, valid_indices, y_train, y_valid)[1]
])
results

,model,input,accuracy,macro_f1,lowest_f1,lowest_f1_label
0,Linear SVC,key and Headers,0.879925,0.844047,0.705882,Psychiatry-Psychology
1,Linear SVC,"desc, key and Headers",0.829268,0.795012,0.651163,Neurosurgery
2,Linear SVC,"desc, sample, key and Headers",0.819887,0.794207,0.651163,Neurosurgery
3,Linear SVC,"sample, key and Headers",0.889306,0.864046,0.777778,Psychiatry-Psychology
4,logistic regression,key and Headers,0.793621,0.752410,0.545455,Dermatology
5,logistic regression,"desc, key and Headers",0.696060,0.690895,0.409091,Neurosurgery
6,logistic regression,"desc, sample, key and Headers",0.694184,0.692605,0.400000,Neurosurgery
7,logistic regression,"sample, key and Headers",0.782364,0.785013,0.666667,Ophthalmology


In [11]:
from collections import Counter

header_counts = Counter(
    header
    for headers in df["transcription_headers"]
    for header in headers
)

header_counts.most_common(20)

[('PROCEDURE', 848),
 ('ANESTHESIA', 839),
 ('PREOPERATIVE DIAGNOSIS', 769),
 ('POSTOPERATIVE DIAGNOSIS', 699),
 ('COMPLICATIONS', 523),
 ('IMPRESSION', 455),
 ('FINDINGS', 441),
 ('ESTIMATED BLOOD LOSS', 420),
 ('PROCEDURE PERFORMED', 415),
 ('POSTOPERATIVE DIAGNOSES', 340),
 ('PREOPERATIVE DIAGNOSES', 302),
 ('HISTORY OF PRESENT ILLNESS', 298),
 ('DESCRIPTION OF PROCEDURE', 276),
 ('HEENT', 271),
 ('EXAM', 268),
 ('INDICATIONS', 263),
 ('ALLERGIES', 253),
 ('MEDICATIONS', 248),
 ('PHYSICAL EXAMINATION', 246),
 ('PLAN', 242)]

In [12]:
sem_kw = df["keywords"] == ""
print(sem_kw.groupby(df["medical_specialty"]).mean().sort_values(ascending=False).round(2))

medical_specialty
Psychiatry-Psychology       0.66
General Medicine            0.42
Neurology                   0.27
Cardiovascular-Pulmonary    0.25
Dermatology                 0.20
Obstetrics-Gynecology       0.17
Orthopedic                  0.16
Neurosurgery                0.15
Gastroenterology            0.15
Radiology                   0.09
Surgery                     0.07
Ophthalmology               0.04
Name: keywords, dtype: float64


In [13]:
sem_kw = (df.loc[valid_indices, "keywords"] == "").to_numpy()
certo = (pred1 == y_valid.to_numpy())
print("com keywords:", certo[~sem_kw].mean(), "| sem keywords:", certo[sem_kw].mean())

com keywords: 0.9887892376681614 | sem keywords: 0.3218390804597701
